# Adaptive-RAG routed baseline: classifier training + evaluation (RQ1)

Paper: Jeong et al., NAACL 2024 (arXiv:2403.14403).
Code: https://github.com/starsuzi/Adaptive-RAG

Trains the t5-large query-complexity classifier and scores the routed
Adaptive-RAG system on the 6 QA test sets. These are the numbers my zero-shot
LLM router is compared against (RQ1).

Runs on a Colab GPU runtime attached from VS Code. The training cell needs
an A100 namely the authors train t5-large at batch size 32 and sequence
length 384, which OOMs on a T4's 15 GB (tried 17 Aug, died 2 steps in).
Keeping their batch size on a bigger card beats shrinking it, so the
reproduction stays faithful to the paper. Everything is re-created on a
fresh runtime: run the cells top to bottom.

In [1]:
![ -d Adaptive-RAG ] || git clone -q https://github.com/starsuzi/Adaptive-RAG.git
%cd Adaptive-RAG
# record the commit I'm working from, this goes in the thesis write-up
COMMIT = !git rev-parse HEAD
print("using commit:", COMMIT[0])

using commit: 0c88670af8707667eb5c1163151bb5ce61b14acb


In [2]:
%%bash
set -e
# colab exports a PYTHONPATH for its own python 3.12, keep it away from the 3.8 env
unset PYTHONPATH
# upstream needs python 3.8 (torch<2 and an old transformers commit, see requirements.txt).
# colab's default python is too new for those pins, so everything runs through a
# small conda env instead. -u lets the installer rerun over an existing install.
wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
bash /tmp/miniconda.sh -b -u -p /opt/miniconda
# python 3.8 from conda-forge (anaconda's default channels now need a ToS acceptance)
[ -d /opt/miniconda/envs/arag ] || \
  /opt/miniconda/bin/conda create -y -q -n arag -c conda-forge --override-channels python=3.8
/opt/miniconda/envs/arag/bin/pip install -q -r requirements.txt
/opt/miniconda/envs/arag/bin/python -c "import _jsonnet; print('env ok')"

PREFIX=/opt/miniconda
Unpacking bootstrapper...
Unpacking payload...

Installing base environment...

Preparing transaction: ...working... done
Executing transaction: ...working... done
installation finished.
Jupyter detected...
Retrieving notices: ...working... done
Channels:
 - conda-forge
Platform: linux-64
Solving environment: ...working... done

## Package Plan ##

  environment location: /opt/miniconda/envs/arag

  added / updated specs:
    - python=3.8


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    _openmp_mutex-4.5          |           20_gnu          28 KB  conda-forge
    bzip2-1.0.8                |      hda65f42_10         252 KB  conda-forge
    ca-certificates-2026.7.22  |       hbd8a1cb_0         129 KB  conda-forge
    icu-78.3                   |  py310h44b86e0_2        13.8 MB  conda-forge
    ld_impl_linux-64-2.46.1    |default_hbd61a6d_102         728 KB  conda-fo

WARNING conda.conda_pypi.main:notify_externally_managed_future(156): 
  Did you know? You can install many PyPI packages with conda
  using the conda-pypi beta. Get started:
    https://docs.conda.io/projects/conda/en/stable/new-features.html



In [3]:
%%bash
# shipped tarballs: per-strategy predictions (the classifier routes between them),
# test subsamples (ground truths), preprocessed classifier training data
tar -xzf predictions.tar.gz
tar -xzf processed_data.tar.gz
tar -xzf data.tar.gz
ls classifier/data | head -3

musique_hotpot_wiki2_nq_tqa_sqd


In [4]:
%%bash
unset PYTHONPATH
# cuda build of the pinned torch
/opt/miniconda/envs/arag/bin/pip install -q torch==1.13.1+cu117 --extra-index-url https://download.pytorch.org/whl/cu117
/opt/miniconda/envs/arag/bin/python -c "import torch; print(torch.cuda.is_available(), torch.cuda.get_device_name(0))"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 GB 17.0 MB/s eta 0:00:00
True NVIDIA A100-SXM4-80GB


In [5]:
%%bash
export PATH=/opt/miniconda/envs/arag/bin:$PATH
# the env ships its own libstdc++ and sqlite; without this the loader picks
# colab's older system libstdc++ and nltk's sqlite import dies on CXXABI
export LD_LIBRARY_PATH=/opt/miniconda/envs/arag/lib:${LD_LIBRARY_PATH:-}
unset PYTHONPATH
cd classifier
# two edits before training: the script pins GPU 7 (authors' server, colab has one gpu),
# and it sweeps epochs 15-35, training five times. the paper settled on epoch 25,
# so one training run is enough here.
sed -i 's/^GPU=7/GPU=0/' run/run_large_train_xl.sh
sed -i 's/^for EPOCH in 15 20 25 30 35/for EPOCH in 25/' run/run_large_train_xl.sh
bash run/run_large_train_xl.sh

{'final_acc_score': 52.22222222222223}
{'A (zero) acc': 24.601366742596813, 'B (single) acc': 64.6888567293777, 'C (multi) acc': 68.18181818181817, 'A (zero) pred num': 191, 'B (single) pred num': 714, 'C (multi) pred num': 445, 'A (zero) gold num': 439, 'B (single) gold num': 691, 'C (multi) gold num': 220}
{'final_acc_score': 0.0}
{'A (zero) acc': -1, 'B (single) acc': -1, 'C (multi) acc': -1, 'A (zero) pred num': 235, 'B (single) pred num': 1567, 'C (multi) pred num': 1198, 'A (zero) gold num': 0, 'B (single) gold num': 0, 'C (multi) gold num': 0}


You're running a t5 model but didn't provide a source prefix, which is the expected, e.g. with `--source_prefix 'summarize: ' `
Generating train split: 3692 examples [00:00, 95940.49 examples/s]
/opt/miniconda/envs/arag/lib/python3.8/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
loading configuration file config.json from cache at /content/Adaptive-RAG/cache/models--t5-large/snapshots/150ebc2c4b72291e770f58e6057481c8d2ed331a/config.json
Model config T5Config {
  "_name_or_path": "t5-large",
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "d_ff": 4096,
  "d_kv": 64,
  "d_model": 1024,
  "decoder_start_token_id": 0,
  "dense_act_fn": "relu",
  "dropout_rate": 0.1,
  "eos_token_id": 1,
  "feed_forward_proj": "relu",
  "initializer_factor": 1.0,
  "is_encoder_decoder": 

In [ ]:
%%bash
export PATH=/opt/miniconda/envs/arag/bin:$PATH
export LD_LIBRARY_PATH=/opt/miniconda/envs/arag/lib:${LD_LIBRARY_PATH:-}
unset PYTHONPATH
# evaluate_final_acc.py scores the multi-hop sets with each dataset's official
# evaluator, which reads the RAW dev files. upstream's download/raw_data.sh
# fetches those plus multi-GB wikipedia corpora; only these pieces are needed:
[ -d official_evaluation/musique ] || bash download/official_eval.sh
pip -q install ujson gdown
mkdir -p raw_data/hotpotqa raw_data/2wikimultihopqa raw_data/musique .temp
[ -f raw_data/hotpotqa/hotpot_dev_distractor_v1.json ] || \
  wget -q http://curtis.ml.cmu.edu/datasets/hotpot/hotpot_dev_distractor_v1.json \
    -O raw_data/hotpotqa/hotpot_dev_distractor_v1.json
if [ ! -f raw_data/2wikimultihopqa/dev.json ]; then
  wget -q "https://www.dropbox.com/s/7ep3h8unu2njfxv/data_ids.zip?dl=1" -O .temp/2wiki.zip
  unzip -joq .temp/2wiki.zip "*dev.json" "*id_aliases.json" -d raw_data/2wikimultihopqa
fi
if [ ! -f raw_data/musique/musique_ans_v1.0_dev.jsonl ]; then
  gdown -q "1tGdADlNjWFaHLeZZGShh2IRcpO6Lv24h&confirm=t" -O .temp/musique_v1.0.zip
  unzip -joq .temp/musique_v1.0.zip "*musique_ans_v1.0_dev.jsonl" -d raw_data/musique
fi
ls -la raw_data/hotpotqa raw_data/2wikimultihopqa raw_data/musique


In [6]:
%%bash
export PATH=/opt/miniconda/envs/arag/bin:$PATH
# the env ships its own libstdc++ and sqlite; without this the loader picks
# colab's older system libstdc++ and nltk's sqlite import dies on CXXABI
export LD_LIBRARY_PATH=/opt/miniconda/envs/arag/lib:${LD_LIBRARY_PATH:-}
unset PYTHONPATH
# both scripts hardcode the authors' timestamped run directory. find the run I just
# trained and point them at it instead.
RESULT=$(ls -t classifier/outputs/*/model/t5-large/flan_t5_xl/epoch/*/*/*/predict/dict_id_pred_results.json | head -1)
echo "using $RESULT"
sed -i "s|^classification_result_file = .*|classification_result_file = './$RESULT'|" classifier/postprocess/predict_complexity_on_classification_results.py
python classifier/postprocess/predict_complexity_on_classification_results.py flan_t5_xl
BASE="predictions/classifier/$(echo "$RESULT" | sed 's|.*/model/||; s|/predict/.*||')/"
sed -i "s|^base_pred_path = .*|base_pred_path = './$BASE'|" evaluate_final_acc.py
python evaluate_final_acc.py

using classifier/outputs/musique_hotpot_wiki2_nq_tqa_sqd/model/t5-large/flan_t5_xl/epoch/25/2026_08_17/04_00_12/predict/dict_id_pred_results.json
predictions/classifier/t5-large/flan_t5_xl/epoch/25/2026_08_17/04_00_12/musique/musique.json
predictions/classifier/t5-large/flan_t5_xl/epoch/25/2026_08_17/04_00_12/musique/musique_option.json
StepNum
musique: 1626
predictions/classifier/t5-large/flan_t5_xl/epoch/25/2026_08_17/04_00_12/hotpotqa/hotpotqa.json
predictions/classifier/t5-large/flan_t5_xl/epoch/25/2026_08_17/04_00_12/hotpotqa/hotpotqa_option.json
StepNum
hotpotqa: 2017
predictions/classifier/t5-large/flan_t5_xl/epoch/25/2026_08_17/04_00_12/2wikimultihopqa/2wikimultihopqa.json
predictions/classifier/t5-large/flan_t5_xl/epoch/25/2026_08_17/04_00_12/2wikimultihopqa/2wikimultihopqa_option.json
StepNum
2wikimultihopqa: 1469
predictions/classifier/t5-large/flan_t5_xl/epoch/25/2026_08_17/04_00_12/nq/nq.json
predictions/classifier/t5-large/flan_t5_xl/epoch/25/2026_08_17/04_00_12/nq/nq_opt

Traceback (most recent call last):
  File "evaluate_final_acc.py", line 339, in <module>
    official_evaluate_by_dicts(data_name)
  File "evaluate_final_acc.py", line 271, in official_evaluate_by_dicts
    original_data = read_jsonl(os.path.join("raw_data", "musique", "musique_ans_v1.0_dev.jsonl"))
  File "/content/Adaptive-RAG/lib.py", line 101, in read_jsonl
    with open(file_path, "r") as file:
FileNotFoundError: [Errno 2] No such file or directory: 'raw_data/musique/musique_ans_v1.0_dev.jsonl'


CalledProcessError: Command 'b'export PATH=/opt/miniconda/envs/arag/bin:$PATH\n# the env ships its own libstdc++ and sqlite; without this the loader picks\n# colab\'s older system libstdc++ and nltk\'s sqlite import dies on CXXABI\nexport LD_LIBRARY_PATH=/opt/miniconda/envs/arag/lib:${LD_LIBRARY_PATH:-}\nunset PYTHONPATH\n# both scripts hardcode the authors\' timestamped run directory. find the run I just\n# trained and point them at it instead.\nRESULT=$(ls -t classifier/outputs/*/model/t5-large/flan_t5_xl/epoch/*/*/*/predict/dict_id_pred_results.json | head -1)\necho "using $RESULT"\nsed -i "s|^classification_result_file = .*|classification_result_file = \'./$RESULT\'|" classifier/postprocess/predict_complexity_on_classification_results.py\npython classifier/postprocess/predict_complexity_on_classification_results.py flan_t5_xl\nBASE="predictions/classifier/$(echo "$RESULT" | sed \'s|.*/model/||; s|/predict/.*||\')/"\nsed -i "s|^base_pred_path = .*|base_pred_path = \'./$BASE\'|" eval

Routed EM/F1 per dataset from the cell above go into the RQ1 comparison table
(zero-shot LLM router vs trained t5-large classifier), alongside routing
accuracy once my router runs on the same test subsamples.